In [ ]:
# Full Pipeline Runner (Databricks)
# Runs Bronze -> Silver -> Gold, then prints expected-vs-actual validation report.

from pyspark.sql import functions as F

dbutils.widgets.text(
    "repo_path",
    "/Workspace/Users/himanshu.kumar1@tothenew.com/databricks-medallion-pipeline",
    "Repo path",
)
repo_path = dbutils.widgets.get("repo_path").strip()

if not repo_path:
    raise ValueError("repo_path widget is required")

print(f"Using repo path: {repo_path}")

notebooks = [
    f"{repo_path}/notebooks/bronze_runtime_validation",
    f"{repo_path}/notebooks/silver_runtime_validation",
    f"{repo_path}/notebooks/gold_runtime_validation",
]

stage_results = []
for nb in notebooks:
    print(f"Running: {nb}")
    try:
        result = dbutils.notebook.run(
            nb,
            timeout_seconds=3600,
            arguments={"repo_path": repo_path},
        )
        stage_results.append((nb.split("/")[-1], "PASS", str(result)))
    except Exception as exc:
        stage_results.append((nb.split("/")[-1], "FAIL", str(exc)))
        print("\n=== Stage Execution Summary ===")
        for stage_name, status, message in stage_results:
            print(f"{stage_name}: {status} | {message}")
        raise

print("\n=== Stage Execution Summary ===")
for stage_name, status, message in stage_results:
    print(f"{stage_name}: {status} | {message}")

checks = [
    ("bronze.customers row_count", 10000, spark.table("bronze.customers").count()),
    ("bronze.products row_count", 500, spark.table("bronze.products").count()),
    ("bronze.orders row_count", 100000, spark.table("bronze.orders").count()),
    ("silver.customers row_count", 10000, spark.table("silver.customers").count()),
    ("silver.products row_count", 500, spark.table("silver.products").count()),
    ("silver.orders row_count", 100000, spark.table("silver.orders").count()),
    ("gold.sales_by_product row_count", 500, spark.table("gold.sales_by_product").count()),
    ("gold.revenue_by_customer row_count", 9940, spark.table("gold.revenue_by_customer").count()),
    ("gold.customer_segmentation row_count", 4, spark.table("gold.customer_segmentation").count()),
    ("gold.daily_weekly_trends row_count", 2679, spark.table("gold.daily_weekly_trends").count()),
]

latest_metrics = spark.sql(
    """
    SELECT run_id, COUNT(*) AS metric_rows
    FROM silver.dq_metrics
    GROUP BY run_id
    ORDER BY MAX(run_timestamp) DESC
    LIMIT 1
    """
).collect()

if not latest_metrics:
    raise RuntimeError("No rows found in silver.dq_metrics. Silver validation report missing.")

checks.append(("silver.dq_metrics latest run rows", 10, int(latest_metrics[0]["metric_rows"])))

report_rows = []
for check_name, expected, actual in checks:
    status = "PASS" if expected == actual else "FAIL"
    report_rows.append((check_name, expected, actual, status))

report_df = spark.createDataFrame(
    report_rows,
    ["check_name", "expected", "actual", "status"],
)

print("\n=== Validation Report (Expected vs Actual) ===")
display(report_df.orderBy("status", "check_name"))

failed_count = report_df.filter(F.col("status") == "FAIL").count()
if failed_count > 0:
    raise RuntimeError(f"Validation failed: {failed_count} check(s) failed.")

print("\nAll validation checks passed.")
